# Data Leakage (fuga de datos)
La fuga de datos es un error bastante comun cuando se trata de predecir un resultado. Eso sucede porque cuando se hace una prediccion nueva la informacion no es como la que se necesita. Por ejemplo, cuando se le entrena a un modelo con 10 columnas/funciones pero cuando se hace una prediccion nueva solo se tienen 7, ahi hay una fuga de datos y la prediccion no sera acertada. Tambien existe cuando la informacion que se esta utilizando para entrenar el modelo contamina la informacion de validacion, en otras palabras, se utiliza la informacion del futuro para predecir el pasado.

Por ejemplo, vamos a utilizar un dataset para una aplicacion de Tarjeta de Credito, pero, no haremos la inspeccion al dataset correspondiente, sino que directamente utilizaremos la informacion que nos dan.

In [12]:
import pandas as pd

data = pd.read_csv("data/AER_credit_card_data.csv", true_values = ['yes'], false_values = ['no'])

y = data.card

X = data.drop(['card'], axis = 1)

print("numero de filas en al dataset: ", X.shape[0])
X.head()

numero de filas en al dataset:  1319


,reports,age,income,share,expenditure,owner,selfemp,dependents,months,majorcards,active
0,0,37.66667,4.5200,0.033270,124.983300,True,False,3,54,1,12
1,0,33.25000,2.4200,0.005217,9.854167,False,False,3,34,1,13
2,0,33.66667,4.5000,0.004156,15.000000,True,False,4,58,1,5
3,0,30.50000,2.5400,0.065214,137.869200,False,False,0,25,1,7
4,0,32.16667,9.7867,0.067051,546.503300,True,False,2,64,1,5


Utilizamos la validacion cruzada para obtener resultados precisos.

In [13]:
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

my_pipeline = make_pipeline(RandomForestClassifier(n_estimators=100))
cv_scores = cross_val_score(my_pipeline, X, y, cv=5, scoring='accuracy')

print("Cross-validation accuracy: %f" % cv_scores.mean())


Cross-validation accuracy: 0.979534


Aquie empezamos "bien". Vemos que la exactiud de nuestro modelo es bastante bueno sin que hicieramos unalimpieza de datos ni que escogieramos los atriutos que mejor nos conviene. Pero aqui hay un error, hay funciones que las personas no nos daran, como la cantidad de dindero que pueden gastar

In [21]:
expenditures_cardholders = X.expenditure[y]
expenditures_noncardholders = X.expenditure[~y]

print('Fraction of those who did not receive a card and had no expenditures: %.2f' \
      %((expenditures_noncardholders == 0).mean()))
print('Fraction of those who received a card and had no expenditures: %.2f' \
      %(( expenditures_cardholders == 0).mean()))

Fraction of those who did not receive a card and had no expenditures: 1.00
Fraction of those who received a card and had no expenditures: 0.02


Como se ve en los resultados, de aquellos que no hacian gastos solo el 2% recibio una tarjeta, mientras que los demas no.

Ahora si elejimos bien los atributos que mejor se pueden explicar.

In [24]:
# eliminamos los predictores del dastaset
potential_leaks = ['expenditure', 'share', 'active', 'majorcards']
X2 = X.drop(potential_leaks, axis = 1)

# evaluamos el modelo con predicores leaky removidos
cv_scores = cross_val_score(my_pipeline, X2, y, cv = 5, scoring = 'accuracy')

print("Cross_val accuracy: %f" % cv_scores.mean())

Cross_val accuracy: 0.827886


Ahora vemos que la exatcitud que le dimos no es tan buena como la anterior, pero podriamos asegurar que es mas confiable.